# Day 5: End-to-End Grounded RAG Pipeline
### Learning Objectives & Success Criteria
- Chain Qdrant Top-K retrieval with externalized prompt hydration (`day05_rag_grounded.yaml`).
- Ground SLM remediation advice strictly in retrieved SOPs to prevent command hallucinations.
- Audit retrieved context vs. generated output in Phoenix.


### Cell 1 — Centralized Settings & Modular Engine Bootstrap


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json, time
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

from engine.settings import SETTINGS
from engine.router import ModelRouter
from engine.qdrant_store import QdrantVectorStore
from engine.use_case_loader import UseCaseLoader
from engine.visualizer import StoryVisualizer
from engine.debugger import TraceDebugger
from engine.quiz_helper import render_quiz

# Import PromptRegistry safely regardless of module filename (prompts.py or prompt_registry.py)
try:
    from engine.prompts import PromptRegistry
except ImportError:
    from engine.prompt_registry import PromptRegistry

# All URLs, paths, domain names, and cloud toggles come from SETTINGS (.env + configs/app_config.yaml)
# To test your own domain, see CUSTOM_USE_CASE_GUIDE.md and set ACTIVE_DOMAIN = "your_domain"
ACTIVE_DOMAIN = SETTINGS.active_domain

loader = UseCaseLoader(domain=ACTIVE_DOMAIN)
llm = ModelRouter(local_model_path=SETTINGS.local_model_path, enable_vertex=SETTINGS.enable_vertex_ai)
qdrant = QdrantVectorStore(collection_name=loader.collection_name, path=SETTINGS.qdrant_db_path)
prompts = PromptRegistry()
viz = StoryVisualizer()
dbg = TraceDebugger(debug_mode=False, enable_phoenix=False)

RUNBOOKS = loader.load_runbooks()
INCIDENTS = loader.load_incidents()
qdrant.upsert_documents(RUNBOOKS)

print(f"[Ready] Domain: {loader.domain} | LLM: {llm.provider} | VectorDB: {qdrant.mode} ({loader.collection_name})")


### Cell 2 — Production Incident Selector (0..9)


In [ ]:
# Select any of the 10 production incidents in the active domain (0..9)
INCIDENT_INDEX = 0
ACTIVE_INCIDENT = INCIDENTS[INCIDENT_INDEX]

print(f"Selected Incident: [{ACTIVE_INCIDENT['incident_id']}] {ACTIVE_INCIDENT['title']} (Severity: {ACTIVE_INCIDENT['severity']})")
print("Raw Log:", ACTIVE_INCIDENT["raw_log"])


### Cell 3 — Engineering Concept Brief
**Concept: Retrieval-Augmented Generation Grounded in Runbooks**
Retrieves top-2 domain runbooks from Qdrant and injects them into `configs/prompts/day05_rag_grounded.yaml`.


### Cell 4 — Traced Execution & X-Ray Visual Storyboard


In [ ]:
dbg.start_run("day05_rag")
with dbg.start_span(f"Day5_CoreRAG_{ACTIVE_INCIDENT['incident_id']}", span_kind="CHAIN") as root_span:
    t0 = time.perf_counter()
    retrieved_docs = qdrant.search(query=ACTIVE_INCIDENT["raw_log"], top_k=2)
    ret_ms = round((time.perf_counter() - t0) * 1000, 2)
    dbg.record_step("qdrant_top2_retrieve", ACTIVE_INCIDENT["raw_log"], retrieved_docs, ret_ms, span_kind="RETRIEVER")

    p_cfg = prompts.load("day05_rag_grounded")
    ctx_block = "\n".join(f"- {d['text']}" for d in retrieved_docs)
    hydrated_prompt = prompts.render(p_cfg["user_template"], retrieved_context=ctx_block, incident_text=ACTIVE_INCIDENT["raw_log"])
    gen = llm.generate(prompt=hydrated_prompt, system_instruction=p_cfg["system_prompt"], temperature=0.1, max_tokens=180)
    dbg.record_step("qwen_grounded_generate", hydrated_prompt, gen["output"], gen["latency_ms"], span_kind="LLM")

viz.render_story(
    title=f"Day 5 Grounded RAG Storyboard — {ACTIVE_INCIDENT['incident_id']}",
    stages=[
        {"name": "Qdrant Top-2 Retrieval", "badge": "RETRIEVER", "latency_ms": ret_ms, "summary": f"Collection: {loader.collection_name}", "scores": [{"label": d["metadata"]["runbook_id"], "score": d["score"]} for d in retrieved_docs], "payload": [d["text"] for d in retrieved_docs]},
        {"name": "Grounded SLM Remediation", "badge": "LLM OUTPUT", "latency_ms": gen["latency_ms"], "summary": "Grounded strictly in retrieved SOPs", "payload": gen["output"]}
    ],
    phoenix_url=dbg.phoenix_url
)


### Cell 5 — Guarded Cloud Parity Check (`SETTINGS.enable_vertex_ai`)


In [ ]:
if SETTINGS.enable_vertex_ai:
    print("Vertex AI Vector Search + Gemini RAG active.")
else:
    print("ENABLE_VERTEX_AI=false -> Local Qdrant + Qwen RAG verified.")


### Cell 6 — Automated Validation Gate


In [ ]:
assert any(d["metadata"]["runbook_id"] == ACTIVE_INCIDENT["expected_runbook"] for d in retrieved_docs)
print("[PASS] Day 5 Grounded RAG Gate passed!")


### Cell 7 — Active Lab Experiment
**Hands-On Lab Challenge:** Switch `ACTIVE_DOMAIN = "payment_ops"` (or `"cicd_devops"`) in Cell 1, restart & run Cells 1–6, and observe how the exact same RAG code retrieves domain-specific runbooks without changing a single line of pipeline logic.


### Cell 8 — Interactive Self-Assessment Questionnaire


In [ ]:
import json, os, sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from engine.quiz_helper import render_quiz

with open(repo_root / "configs" / "quizzes" / "day05.json", "r", encoding="utf-8") as f:
    quiz_cfg = json.load(f)

render_quiz(quiz_cfg["title"], quiz_cfg["questions"])


### Cell 9 — Bring Your Own Use Case (BYOUC)
Want to run all 10 notebooks against your own enterprise domain? Follow **[CUSTOM_USE_CASE_GUIDE.md](../CUSTOM_USE_CASE_GUIDE.md)** (`python scripts/scaffold_domain.py --domain your_domain --prefix YOUR-`) and set `ACTIVE_DOMAIN = "your_domain"` in Cell 1.
